# X/Y 交流控制噪声谱测量

测量 Bell-Bloom 磁力仪的探测光 $S_1$ 的功率谱密度 $S_{S_1}(\omega, \Omega_\text{Ctrl})$，通过二维拟合同时提取原子噪声谱 $S_\beta(\omega)$ 和光子噪声 $N_{S_1}(\omega)$，为 Floquet 最优控制提供输入。

## 原理

扫描 X/Y 补偿线圈的交流控制场幅度，在不同控制强度 $\Omega_\text{Ctrl}$ 下，$S_\beta(\omega)$ 受到洛伦兹滤波函数 $L(\omega, \Omega_\text{Ctrl})$ 的不同调制，而 $N_{S_1}(\omega)$ 保持不变。通过二维拟合实现宽带噪声谱重建。

## 硬件连接

| 信号 | 仪器 | 通道 | 说明 |
|------|------|------|------|
| X 控制 | DG4000 (DG4E234902522) | CH1 | 9kHz Burst 正弦波，幅度扫场 |
| Y 控制 | DG4000 (DG4E234902522) | CH2 | 9kHz Burst 正弦波，X+90° |
| Pump 调制 | DG4000 (DG4E222800868) | CH1+CH2 | 100MHz 正弦 + 脉冲门控 (RF 开关) |
| 主磁场 | GS200 | - | 恒流模式，~9.3 mA |
| Pump 光功率 | DG4000 (DG4E231500376) | CH1 DC | - |
| Probe 光功率 | DG4000 (DG4E231500376) | CH2 DC | - |
| 温度控制 | TEC103 | - | 气室温度控制 |
| 温度开关 | DG4000 (DG4E271200104) | CH2 | TTL 电平控制温控通断 |
| 噪声采集+相位校准 | HF2 锁相 | 信号输入 0 | DAQ 模块采集解调信号 |

## 实验参数概览

- **扫描范围**: XY 幅度 0.01 V → 7.0 V, 700 点
- **控制频率**: 9 kHz, X/Y 相位差 90°
- **DAQ 采集**: 时长 1.0 s, 采样率 ~500 kSa/s
- **每点稳定时间**: 5.0 s

In [ ]:
from pathlib import Path
import sys
# 自动定位项目根目录（以 params/ 目录为标记）
project_root = Path.cwd()
while not (project_root / "params").exists() and project_root.parent != project_root:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import yaml
import time
import json
from datetime import datetime
import matplotlib.pyplot as plt

# 设备库
from gs200 import GS200Instrument
from signal_generator import DG4000Instrument
from tec_controller import TECInstrument
from lockin_amplifier import (
    HF2Instrument, DAQConfig, DAQResult,
    SignalInputConfig, OscillatorConfig, DemodulatorConfig,
    demod, daq,
)

# 数据分析库
from scipy import signal as scipy_signal
from scipy.optimize import curve_fit

print("所有库导入成功")

In [ ]:
# 加载物理量→仪器映射
with open(project_root / "params" / "mapping.yaml", encoding="utf-8") as f:
    MAPPING = yaml.safe_load(f)["mapping"]

# 加载安全限值
with open(project_root / "params" / "safety_limits.yaml", encoding="utf-8") as f:
    LIMITS = yaml.safe_load(f)["safety_limits"]

# ========== 实验参数 ==========
EXPERIMENT_TYPE = "Noise_Spectrum_XY_Ctrl"
PURPOSE = "noise_spectroscopy"

# ---- 扫描参数 ----
XY_AMP_START = 0.01        # X/Y 幅度扫描起始 (V)
XY_AMP_STOP = 7.0          # X/Y 幅度扫描终止 (V)
XY_AMP_POINTS = 700        # 扫描点数
XY_SETTLE_TIME = 5.0       # 每点等待稳定时间 (s)

# ---- X/Y 交流控制信号 ----
XY_CTRL_FREQ = 9000        # X/Y 交流控制频率 (Hz)
XY_CTRL_PHASE = 100        # X 控制信号相位 (deg)，Y 相位 = X 相位 + 90°

# ---- Pump 调制参数 ----
PUMP_MOD_FREQ = 90e3       # Pump 调制频率 (Hz)
PUMP_MOD_AMPLITUDE = 0.18  # 100MHz 正弦波幅度 (V)
PUMP_MOD_DUTY = 5          # 脉冲占空比 (%)
RF_GATE_AMPLITUDE = 5.0    # 脉冲门控幅度 (Vpp)
RF_GATE_OFFSET = 2.5       # 脉冲门控偏置 (V)

# ---- HF2 解调配置（相位校准用） ----
HF2_DEMOD_IDX = 0          # 解调器索引
HF2_OSC_FREQ = 90e3        # 振荡器频率 (Hz)，与 Pump 调制频率一致
HF2_SIGNAL_RANGE = 2.0     # 信号输入量程 (V)
HF2_DEMOD_ORDER = 4        # 解调滤波器阶数
HF2_DEMOD_TC = 0.000692    # 解调时间常数 (s)
HF2_DEMOD_RATE = 500000    # 解调输出数据速率 (Sa/s)

# ---- HF2 DAQ 采集配置（噪声采集用） ----
HF2_DAQ_DURATION = 1.0     # DAQ 采集时长 (s)
HF2_NPERSEG = 20000        # Welch PSD 每段点数

# ---- 固定参数 ----
FIXED_PARAMS = {
    "Pump_laser_power": 1.0,         "Probe_laser_power": 1.0,
    "temperature": 100,              "Temp_Switch": 5.0,
    "main_magnetic_field": 9.285,
}

# ---- 运行目录命名 ----
RUN_TAG = "noise"

print("配置已加载")

In [ ]:
# 安全边界检查函数
def validate_safety_limit(name, value):
    """检查数值是否在安全范围内，超出则报错."""
    lim = LIMITS.get(name)
    if lim is None:
        return value
    lo, hi = lim["min"], lim["max"]
    if lo is not None and hi is not None:
        if value < lo or value > hi:
            raise ValueError(
                f"[安全拦截] {name}={value} 超出范围 [{lo}, {hi}]"
            )
    return value

print("安全校验函数已定义")

In [ ]:
devices = {}

try:
    # ---- GS200: 主磁场 ----
    gs_cfg = MAPPING["main_magnetic_field"]
    gs = GS200Instrument(gs_cfg["resource"])
    gs.connect()
    print(f"GS200 已连接: {gs.idn()}")
    gs.set_source_function(gs_cfg["source_function"])
    # [经验] GS200 需硬件级电流保护
    gs.set_current_limit(LIMITS["main_magnetic_field"]["max"] / 1000.0)
    devices["gs200"] = gs

    # ---- DG4000: X/Y 补偿磁场（Burster） ----
    dg_comp_cfg = MAPPING["X_magnetic_field"]
    dg_comp = DG4000Instrument(dg_comp_cfg["resource"], channel=1)
    dg_comp.connect()
    print(f"补偿场 DG4000 已连接: {dg_comp.idn()}")
    dg_comp.set_ref_clock_source("EXTernal")
    # Y 通道: 通过同一设备 CH2 控制
    devices["dg_comp"] = dg_comp

    # ---- DG4000: Pump 调制 ----
    dg_mod_cfg = MAPPING["Pump_modulation"]
    dg_mod = DG4000Instrument(dg_mod_cfg["resource"], channel=1)
    dg_mod.connect()
    print(f"调制 DG4000 已连接: {dg_mod.idn()}")
    devices["dg_mod"] = dg_mod

    # ---- DG4000: 温度开关 ----
    dg_temp_cfg = MAPPING["Temp_Switch"]
    dg_temp = DG4000Instrument(dg_temp_cfg["resource"], channel=2)
    dg_temp.connect()
    print(f"温控 DG4000 已连接: {dg_temp.idn()}")
    devices["dg_temp"] = dg_temp

    # ---- TEC103: 温度控制器 ----
    tec_cfg = MAPPING["temperature"]
    tec = TECInstrument(port=tec_cfg["resource"])
    tec.connect()
    print(f"TEC103 已连接")
    devices["tec"] = tec

    # ---- DG4000: Pump/Probe 光功率 ----
    dg_laser_cfg = MAPPING["Pump_laser_power"]
    dg_laser = DG4000Instrument(dg_laser_cfg["resource"], channel=1)
    dg_laser.connect()
    print(f"光功率 DG4000 已连接: {dg_laser.idn()}")
    devices["dg_laser"] = dg_laser

    # ---- HF2: 锁相放大器 ----
    hf2_cfg = MAPPING["lockin_r"]
    hfi = HF2Instrument(
        host=hf2_cfg.get("host", "127.0.0.1"),
        port=hf2_cfg.get("port", 8005),
        api_level=1,
        device_id=hf2_cfg["device_id"],
    )
    hfi.connect()
    print(f"HF2 已连接: {hfi.idn}")
    hfi.set_extclk(True)
    print(f"  -> HF2 时钟源: 外部")
    devices["hf2"] = hfi

except Exception as e:
    print(f"设备连接失败: {e}")
    raise

print(f"\n所有设备连接完成，共 {len(devices)} 个设备")

In [ ]:
hfi = devices["hf2"]
tec = devices["tec"]
gs = devices["gs200"]
dg_laser = devices["dg_laser"]
dg_comp = devices["dg_comp"]
dg_mod = devices["dg_mod"]
dg_temp = devices["dg_temp"]

# ---- 1. Pump 光功率 ----
validate_safety_limit("Pump_laser_power", FIXED_PARAMS["Pump_laser_power"])
dg_laser.setup_dc(FIXED_PARAMS["Pump_laser_power"], channel=1)
print(f"Pump 光功率: {FIXED_PARAMS['Pump_laser_power']} V DC")

# ---- 2. Probe 光功率 ----
validate_safety_limit("Probe_laser_power", FIXED_PARAMS["Probe_laser_power"])
dg_laser.setup_dc(FIXED_PARAMS["Probe_laser_power"], channel=2)
print(f"Probe 光功率: {FIXED_PARAMS['Probe_laser_power']} V DC")

# ---- 3. 主磁场 ----
validate_safety_limit("main_magnetic_field", FIXED_PARAMS["main_magnetic_field"])
gs.set_current(FIXED_PARAMS["main_magnetic_field"] / 1000.0)
gs.set_output(True)
print(f"主磁场: {FIXED_PARAMS['main_magnetic_field']} mA")

# ---- 4. 温度开关 (ON) ----
validate_safety_limit("Temp_Switch", FIXED_PARAMS["Temp_Switch"])
dg_temp.setup_dc(FIXED_PARAMS["Temp_Switch"], channel=2)
dg_temp.set_output(True, channel=2)
print(f"温度开关: ON ({FIXED_PARAMS['Temp_Switch']} V)")

# ---- 5. 温度控制 ----
validate_safety_limit("temperature", FIXED_PARAMS["temperature"])
tec.set_target_temperature(FIXED_PARAMS["temperature"], channel=1)
tec.set_enable(True, channel=1)
temp_now = tec.get_temperature(channel=1)
print(f"温度设定: {FIXED_PARAMS['temperature']} °C, 当前: {temp_now:.1f} °C")
print("等待温度稳定...")
while True:
    time.sleep(5)
    t = tec.get_temperature(channel=1)
    print(f"  当前温度: {t:.2f} °C")
    try:
        t2 = tec.get_temperature(channel=1)
        if abs(t2 - FIXED_PARAMS['temperature']) < 1:
            print(f"温度已稳定: {t:.2f} °C")
            break
    except:
        pass

# ---- 创建运行目录 ----
timestamp = datetime.now().strftime("%m%d_%H%M")
run_dir = project_root / "data" / EXPERIMENT_TYPE / f"{timestamp}_{RUN_TAG}"
run_dir.mkdir(parents=True, exist_ok=True)
(raw_dir := run_dir / "raw").mkdir(exist_ok=True)
(results_dir := run_dir / "results").mkdir(exist_ok=True)
print(f"运行目录: {run_dir}")

# ---- 保存实验配置到运行目录 ----
config = {
    "experiment_type": EXPERIMENT_TYPE,
    "purpose": PURPOSE,
    "timestamp": timestamp,
    "scan_params": {
        "XY_AMP_START_V": XY_AMP_START,
        "XY_AMP_STOP_V": XY_AMP_STOP,
        "XY_AMP_POINTS": XY_AMP_POINTS,
        "XY_SETTLE_TIME_s": XY_SETTLE_TIME,
    },
    "xy_ctrl": {
        "XY_CTRL_FREQ_Hz": XY_CTRL_FREQ,
        "XY_CTRL_PHASE_deg": XY_CTRL_PHASE,
    },
    "fixed_params": FIXED_PARAMS,
    "pump_modulation": {
        "pump_mod_freq_Hz": PUMP_MOD_FREQ,
        "pump_mod_amplitude_V": PUMP_MOD_AMPLITUDE,
        "pump_mod_duty_pct": PUMP_MOD_DUTY,
    },
    "hf2_demod": {
        "demod_idx": HF2_DEMOD_IDX,
        "osc_freq_Hz": HF2_OSC_FREQ,
        "signal_range_V": HF2_SIGNAL_RANGE,
        "demod_rate_Sa_s": HF2_DEMOD_RATE,
        "demod_TC_s": HF2_DEMOD_TC,
    },
    "hf2_daq": {
        "DAQ_duration_s": HF2_DAQ_DURATION,
        "nperseg": HF2_NPERSEG,
    },
}
config_path = run_dir / "experiment_config.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
print(f"实验配置已保存: {config_path}")

print("\n初始值设置完成")

In [ ]:
# ---- 配置 RF 开关方案（Pump 调制） ----
print("CH1: 配置 100MHz 连续正弦波...")
dg_mod.setup_sine(freq=100e6, amplitude=PUMP_MOD_AMPLITUDE,
                  offset=0.0, phase=0.0, channel=1)
print(f"  100MHz, 幅度 {PUMP_MOD_AMPLITUDE*1000:.0f} mVpp → RF 开关 IN")

pulse_width = (PUMP_MOD_DUTY / 100.0) / PUMP_MOD_FREQ
print(f"\nCH2: 配置 {PUMP_MOD_FREQ/1e3:.0f} kHz 脉冲门控...")
print(f"  脉宽: {pulse_width*1e6:.2f} μs ({PUMP_MOD_DUTY}% duty)")
dg_mod.setup_pulse(freq=PUMP_MOD_FREQ, amplitude=RF_GATE_AMPLITUDE,
                   offset=RF_GATE_OFFSET, width=pulse_width, channel=2)
print(f"  {PUMP_MOD_FREQ/1e3:.0f} kHz → RF 开关 CTRL")

print("\n✅ RF 开关方案配置完成")

# ---- X/Y 控制信号初始配置（Burst 模式） ----
# [经验] 优先使用 DG4000 内置 Burst 模式替代 draw_waveform 下载波形
print("\n配置 X/Y 交流控制信号（Burst 模式）...")

# 设置 CH1: X 通道正弦波
dg_comp.setup_sine(freq=XY_CTRL_FREQ, amplitude=XY_AMP_START,
                   offset=0.0, phase=0.0, channel=1)
# 转为 Burst 模式（INFINITY = 连续输出）
dg_comp.set_burst_state(True, channel=1)
dg_comp.set_burst_mode("INFinity", channel=1)
dg_comp.set_burst_ncycles(50000, channel=1)
dg_comp.set_burst_phase(0.0, channel=1)
dg_comp.set_burst_trigger_source("INTernal", channel=1)
print(f"  X CH1: Burst 模式, {XY_CTRL_FREQ} Hz, 幅度 {XY_AMP_START} V")

# 设置 CH2: Y 通道正弦波，相位 = X + 90°
dg_comp.setup_sine(freq=XY_CTRL_FREQ, amplitude=XY_AMP_START,
                   offset=0.0, phase=XY_CTRL_PHASE + 90.0, channel=2)
dg_comp.set_burst_state(True, channel=2)
dg_comp.set_burst_mode("INFinity", channel=2)
dg_comp.set_burst_ncycles(50000, channel=2)
dg_comp.set_burst_phase(XY_CTRL_PHASE + 90.0, channel=2)
dg_comp.set_burst_trigger_source("INTernal", channel=2)
print(f"  Y CH2: Burst 模式, {XY_CTRL_FREQ} Hz, 幅度 {XY_AMP_START} V, 相位 X+90°")

# 打开输出
dg_comp.set_output(True, channel=1)
dg_comp.set_output(True, channel=2)
print("  X/Y 输出 ON")

# ---- HF2 配置 + 相位校准 ----
print("\n关闭温度开关（相位校准前）...")
dg_temp.set_output(False, channel=2)

# 配置信号输入
sig_in_cfg = SignalInputConfig(
    input_index=0, range=HF2_SIGNAL_RANGE,
    ac_coupling=True, diff=False, impedance=50,
)
demod.configure_signal_input(hfi, sig_in_cfg)
print(f"信号输入已配置: {HF2_SIGNAL_RANGE} V range, AC coupled")

# 配置振荡器
osc_cfg = OscillatorConfig(
    osc_index=0, frequency=HF2_OSC_FREQ, source="manual",
)
demod.configure_oscillator(hfi, osc_cfg)
print(f"振荡器已配置: {HF2_OSC_FREQ/1e3:.0f} kHz")

# 配置解调器
demod_cfg = DemodulatorConfig(
    demod_index=0, enable=True, rate=HF2_DEMOD_RATE,
    input_channel=0, osc_select=0, harmonic=1,
    time_constant=HF2_DEMOD_TC, order=HF2_DEMOD_ORDER, phase=0.0,
)
actual_rate = demod.configure_demodulator(hfi, demod_cfg)
print(f"解调器 0 已配置: rate={actual_rate:.0f} Sa/s, TC={HF2_DEMOD_TC*1000:.3f} ms")

# [经验] 使用 demod.auto_calibrate_phase() 做相位校准，无需复杂反馈闭环
print("正在进行相位校准...")
calibrated_phase = demod.auto_calibrate_phase(
    hfi, demod_idx=0, tolerance_deg=1.0, max_attempts=5, settle_time=0.2
)
print(f"相位校准完成: {calibrated_phase:.2f}°")

# 读一组样本验证
sample = demod.read_demod_sample(hfi, demod_idx=0)
print(f"校准后样本: R={sample['r']:.6f}, X={sample['x']:.6f}, Y={sample['y']:.6f}")

time.sleep(0.5)

# 恢复温控
print("恢复温度开关...")
dg_temp.set_output(True, channel=2)
time.sleep(0.5)

In [ ]:
# ===== 逐点扫描 X/Y 幅度 + HF2 DAQ 采集 =====
# [经验] 扫描循环须用 try/finally 包裹，确保异常时恢复温度开关
# [经验] 每点 X/Y 幅度设置前调用 validate_safety_limit() 做安全限值检查
# [经验] 每次幅度变化后需等待足够时间（5s）让系统稳定

amplitudes = np.linspace(XY_AMP_START, XY_AMP_STOP, XY_AMP_POINTS)
print("=" * 60)
print(f"X/Y 幅度扫描噪声谱测量")
print("=" * 60)
print(f"控制频率: {XY_CTRL_FREQ} Hz")
print(f"扫描范围: {XY_AMP_START} V → {XY_AMP_STOP} V, {XY_AMP_POINTS} 点")
print(f"DAQ 采集: 时长 {HF2_DAQ_DURATION}s, rate={actual_rate:.0f} Sa/s")
total_est = XY_AMP_POINTS * (XY_SETTLE_TIME + HF2_DAQ_DURATION + 2.0)
print(f"预计耗时: {total_est:.0f}s ≈ {total_est/3600:.1f}h")
print()

# [经验] 使用 actual_rate 而非硬编码常量
grid_cols = int(actual_rate * HF2_DAQ_DURATION)
print(f"DAQ grid_cols = {grid_cols} (actual_rate × duration)")

t_start = time.time()

# try/finally 确保异常时恢复温度开关
try:
    for i, amp in enumerate(amplitudes):
        # 安全限值检查
        amp = validate_safety_limit("X_magnetic_field", amp)

        # 设置 X/Y 幅度（Burst 模式下调节幅度）
        dg_comp.set_amplitude(amp, channel=1)
        dg_comp.set_amplitude(amp, channel=2)

        # 关闭温度开关（消除温控磁场干扰）
        dg_temp.set_output(False, channel=2)

        # 等待系统稳定
        time.sleep(XY_SETTLE_TIME)

        # 配置 HF2 DAQ 采集（每点独立配置）
        # [经验] 已改用 HF2 DAQ 模块采集噪声，无需示波器
        daq_cfg = DAQConfig(
            device=MAPPING["lockin_r"]["device_id"],
            trigger_type=0,                  # 连续模式
            duration=HF2_DAQ_DURATION,
            grid_cols=grid_cols,
            grid_rows=1,
            grid_mode=2,
            signal_paths=["sample.r"],       # 采集幅值信号
        )

        # 采集解调时域信号
        try:
            results = daq.acquire_data(
                hfi, daq_cfg, demod_idx=0,
                actual_rate=actual_rate, timeout=HF2_DAQ_DURATION + 10.0,
            )
            waveform = results[0].values
        except Exception as e:
            print(f"  [{i:4d}/{XY_AMP_POINTS}] amp={amp:.4f}V: DAQ 采集失败: {e}")
            waveform = np.array([np.nan])

        # 保存原始波形（.npy 二进制格式）
        # [经验] 用 .npy 替代 CSV 保存波形
        np.save(raw_dir / f"waveform_C{i:04d}.npy", waveform)

        # 恢复温度开关
        dg_temp.set_output(True, channel=2)

        # 进度报告
        elapsed = time.time() - t_start
        progress_pct = (i + 1) / XY_AMP_POINTS * 100
        remain = (total_est - elapsed) if elapsed < total_est else 0
        print(f"  [{progress_pct:5.1f}%] amp={amp:.4f}V, "
              f"pts={len(waveform)}, "
              f"已用 {elapsed:.0f}s, 预计剩余 {remain:.0f}s")

except Exception as e:
    print(f"\n❌ 扫描出错: {e}")
    # 紧急恢复温度开关
    print("紧急恢复温度开关...")
    dg_temp.set_output(True, channel=2)
    raise

finally:
    # [经验] 确保异常退出时温控恢复
    dg_temp.set_output(True, channel=2)
    print("温度开关已恢复 ON")

elapsed_total = time.time() - t_start
print(f"\n✅ 扫描完成！共 {XY_AMP_POINTS} 点, 用时 {elapsed_total:.0f}s")
print(f"原始波形保存在: {raw_dir}")

In [ ]:
# ===== 数据分析：计算 Welch PSD =====
# 可从原始 .npy 文件加载（离线分析），后备用内存变量

# 确定扫描点数
n_files = len(list(raw_dir.glob("waveform_C*.npy")))
print(f"找到 {n_files} 个波形文件")

# [经验] 用 .npz 替代 CSV 保存矩阵数据
# 计算 PSD 矩阵
psd_list = []
freq_axis = None

for i in range(n_files):
    wf_path = raw_dir / f"waveform_C{i:04d}.npy"
    if not wf_path.exists():
        print(f"  ⚠️ 缺失: {wf_path.name}")
        continue

    waveform = np.load(wf_path)

    # 检查数据有效性
    if np.all(np.isnan(waveform)) or len(waveform) < HF2_NPERSEG:
        print(f"  ⚠️ 无效波形: {wf_path.name}, 跳过 PSD 计算")
        psd_list.append(np.full(HF2_NPERSEG // 2 + 1, np.nan))
        if i == 0:
            freq_axis = np.fft.rfftfreq(HF2_NPERSEG, d=1.0/actual_rate)
        continue

    # [经验] 使用 Welch 法计算 PSD
    f, psd = scipy_signal.welch(
        waveform, fs=actual_rate,
        nperseg=min(HF2_NPERSEG, len(waveform)),
        noverlap=None,
        scaling="density",
    )
    if i == 0:
        freq_axis = f
    psd_list.append(psd)

# 构建 PSD 矩阵: shape = (n_amplitudes, n_frequencies)
psd_matrix = np.array(psd_list)
amplitudes_used = np.linspace(XY_AMP_START, XY_AMP_STOP, len(psd_matrix))

print(f"PSD 矩阵形状: {psd_matrix.shape}")
print(f"频率范围: {freq_axis[0]:.0f} - {freq_axis[-1]:.0f} Hz")
print(f"频率点数: {len(freq_axis)}")

# 保存 PSD 矩阵
np.savez(
    results_dir / "psd_matrix.npz",
    psd_matrix=psd_matrix,
    freq_axis=freq_axis,
    amplitudes=amplitudes_used,
    actual_rate=actual_rate,
)
print(f"\nPSD 矩阵已保存: {results_dir / 'psd_matrix.npz'}")

In [ ]:
# ===== 幅频标定：峰检测 + 线性回归 =====
# 控制幅度（V）与有效控制频率 Ω_Ctrl（Hz）之间为线性关系:
#   Ω_Ctrl = k · V + b

# 从 PSD 矩阵中检测峰位置
calib_peaks = []
ctrl_freq_range = (XY_CTRL_FREQ - 500, XY_CTRL_FREQ + 500)  # 搜索范围

for i in range(len(amplitudes_used)):
    psd_segment = psd_matrix[i, :]
    if np.all(np.isnan(psd_segment)):
        calib_peaks.append(np.nan)
        continue

    # 在控制频率附近找峰
    mask = (freq_axis >= ctrl_freq_range[0]) & (freq_axis <= ctrl_freq_range[1])
    if not np.any(mask):
        calib_peaks.append(np.nan)
        continue

    idx_peak = np.nanargmax(psd_segment[mask])
    # 映射回全局索引
    global_idx = np.where(mask)[0][idx_peak]
    calib_peaks.append(freq_axis[global_idx])

calib_peaks = np.array(calib_peaks)

# 线性回归
valid_mask = ~np.isnan(calib_peaks)
if np.sum(valid_mask) > 2:
    k, b = np.polyfit(amplitudes_used[valid_mask], calib_peaks[valid_mask], 1)
    omega_ctrl = k * amplitudes_used + b
    print(f"幅频标定结果:")
    print(f"  k = {k:.4f} Hz/V")
    print(f"  b = {b:.4f} Hz")
    print(f"  Ω_Ctrl = {k:.4f} · V + {b:.4f}")
else:
    k, b = 0, XY_CTRL_FREQ
    omega_ctrl = np.full_like(amplitudes_used, XY_CTRL_FREQ)
    print("⚠️ 峰检测失败，使用 nominal 控制频率")

# 保存标定结果
np.savez(
    results_dir / "calibration.npz",
    k=k, b=b,
    amplitudes=amplitudes_used,
    omega_ctrl=omega_ctrl,
    calib_peaks=calib_peaks,
    valid_mask=valid_mask,
)
print(f"标定结果已保存: {results_dir / 'calibration.npz'}")

In [ ]:
# ===== 洛伦兹拟合提取噪声参数 =====
# 在每个频率点 ω 上，对控制强度 Ω_Ctrl 做洛伦兹拟合
#
# S_S1(Ω_Ctrl; ω) = D + A · (γ² + ω²) / ((γ² - ω² + (Ω_Ctrl + Δω)²)² + 4ω²γ²)
#
# 拟合参数: gamma (γ), Amp (A), D (N_S1), dw (Δω)

def lorentzian_vs_omega(omega_ctrl, gamma, Amp, D, dw, omega_fixed):
    """洛伦兹函数: PSD vs 控制强度，在固定频率 omega_fixed 上

    Parameters
    ----------
    omega_ctrl : array
        控制频率 (Hz)
    gamma : float
        线宽 (Hz)
    Amp : float
        振幅 (V²/Hz)
    D : float
        基线噪声 (V²/Hz)
    dw : float
        频率偏移 (Hz)
    omega_fixed : float
        当前拟合的频率点 (Hz)
    """
    w = omega_fixed
    Om = omega_ctrl + dw
    return D + Amp * (gamma**2 + w**2) / (
        (gamma**2 - w**2 + Om**2)**2 + 4 * w**2 * gamma**2
    )

# 初始化结果数组
n_freq = len(freq_axis)
popt_list = np.full((n_freq, 4), np.nan)  # gamma, Amp, D, dw
perr_list = np.full((n_freq, 4), np.nan)
fit_mask = np.zeros(n_freq, dtype=bool)

# [经验] 拟合前做预筛选（PSD 峰 > 2σ 再做拟合），失败点用插值填充
print("洛伦兹拟合中...")
for j in range(n_freq):
    psd_slice = psd_matrix[:, j]  # 所有幅度点在该频率的 PSD

    if np.all(np.isnan(psd_slice)):
        continue

    # 预筛选：检查该频率点是否有明显共振峰
    median_val = np.nanmedian(psd_slice)
    std_val = np.nanstd(psd_slice)
    max_val = np.nanmax(psd_slice)

    if max_val - median_val <= 2 * std_val:
        continue  # 无明显峰，跳过拟合

    # 有效数据掩码
    valid = ~np.isnan(psd_slice) & ~np.isnan(omega_ctrl)
    if np.sum(valid) < 10:
        continue

    x_data = omega_ctrl[valid]
    y_data = psd_slice[valid]

    # 初始值估计
    gamma_guess = 500.0
    Amp_guess = max_val - median_val
    D_guess = median_val
    dw_guess = 0.0

    try:
        popt, pcov = curve_fit(
            lambda x, g, A, D, dw: lorentzian_vs_omega(x, g, A, D, dw, freq_axis[j]),
            x_data, y_data,
            p0=[gamma_guess, Amp_guess, D_guess, dw_guess],
            maxfev=5000,
        )
        popt_list[j, :] = popt
        perr_list[j, :] = np.sqrt(np.diag(pcov))
        fit_mask[j] = True
    except (RuntimeError, ValueError) as e:
        pass  # 拟合失败，保留 NaN

fitted_count = np.sum(fit_mask)
print(f"拟合完成: {fitted_count}/{n_freq} 频率点拟合成功 ({100*fitted_count/n_freq:.1f}%)")

# [经验] 失败点用插值填充
if fitted_count > 0 and fitted_count < n_freq:
    from scipy.interpolate import interp1d

    for param_idx in range(4):
        valid_idx = np.where(fit_mask)[0]
        if len(valid_idx) > 2:
            interp_func = interp1d(
                valid_idx, popt_list[valid_idx, param_idx],
                kind="linear", fill_value="extrapolate",
            )
            nan_idx = np.where(~fit_mask)[0]
            for idx in nan_idx:
                popt_list[idx, param_idx] = float(interp_func(idx))

    print("失败点已用线性插值填充")

# 提取噪声谱
# S_beta(ω) ∝ Amp(ω), N_S1(ω) = D(ω)
S_beta = popt_list[:, 1].copy()  # Amp 参数
N_S1 = popt_list[:, 2].copy()    # D 参数

# 保存拟合结果
np.savez(
    results_dir / "popt_fit.npz",
    popt=popt_list,
    perr=perr_list,
    fit_mask=fit_mask,
    freq_axis=freq_axis,
    param_names=["gamma", "Amp", "D", "dw"],
)

# [经验] 保存 noise_spectra.npz 供最优控制设计 Notebook 加载
np.savez(
    results_dir / "noise_spectra.npz",
    S_beta=S_beta,
    N_S1=N_S1,
    freq_axis=freq_axis,
    amplitudes=amplitudes_used,
    omega_ctrl=omega_ctrl,
)
print(f"\n噪声谱已保存: {results_dir / 'noise_spectra.npz'}")
print(f"  S_beta (原子噪声): {S_beta.shape}")
print(f"  N_S1 (光子噪声): {N_S1.shape}")
print(f"  freq_axis: {freq_axis[0]:.0f} - {freq_axis[-1]:.0f} Hz")

In [ ]:
# ===== 绘制结果图 =====

# ---- 图 1: PSD 二维伪彩图 ----
fig1, ax1 = plt.subplots(figsize=(10, 6))
# 取 PSD 的对数
psd_log = np.log10(np.maximum(psd_matrix, 1e-20))
extent = [freq_axis[0], freq_axis[-1],
          amplitudes_used[0], amplitudes_used[-1]]
im = ax1.imshow(psd_log, aspect="auto", origin="lower",
                extent=extent, cmap="inferno")
ax1.set_xlabel("Frequency (Hz)")
ax1.set_ylabel("Control Amplitude (V)")
ax1.set_title("PSD $S_{S_1}(\\omega, \\Omega_\\mathrm{Ctrl})$")
cb1 = fig1.colorbar(im, ax=ax1, label="$\\log_{10}$ PSD (V²/Hz)")
fig1.tight_layout()
fig1.savefig(results_dir / "noise_spectrum_2d.png", dpi=150)
print(f"PSD 伪彩图已保存")

# ---- 图 2: 典型幅度的 PSD 及拟合示例 ----
fig2, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# 选择 4 个典型幅度点
n_amps = len(amplitudes_used)
idx_samples = [0, n_amps // 3, n_amps * 2 // 3, n_amps - 1]

for idx_ax, idx_amp in enumerate(idx_samples):
    ax = axes[idx_ax]
    amp = amplitudes_used[idx_amp]

    # 绘制原始 PSD
    ax.loglog(freq_axis, psd_matrix[idx_amp, :], "b-",
              alpha=0.7, label=f"A = {amp:.2f} V")

    # 绘制洛伦兹拟合（用提取的参数重建）
    omega_ctrl_this = omega_ctrl[idx_amp]
    psd_reconstruct = lorentzian_vs_omega(
        np.full(1, omega_ctrl_this),
        popt_list[:, 0], popt_list[:, 1],
        popt_list[:, 2], popt_list[:, 3],
        freq_axis,
    )
    ax.loglog(freq_axis, psd_reconstruct, "r--",
              alpha=0.5, label="fit")

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD (V²/Hz)")
    ax.set_title(f"PSD at A = {amp:.2f} V")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig2.tight_layout()
fig2.savefig(results_dir / "noise_spectrum_fit.png", dpi=150)
print(f"拟合示例图已保存")

# ---- 图 3: 提取的噪声谱 ----
fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(14, 5))

ax3a.loglog(freq_axis, S_beta, "b-", linewidth=1.5)
ax3a.set_xlabel("Frequency (Hz)")
ax3a.set_ylabel("$S_\\beta(\\omega)$ (V²/Hz)")
ax3a.set_title("Atomic Noise Spectrum $S_\\beta(\\omega)$")
ax3a.grid(True, alpha=0.3)

ax3b.loglog(freq_axis, N_S1, "r-", linewidth=1.5)
ax3b.set_xlabel("Frequency (Hz)")
ax3b.set_ylabel("$N_{S_1}(\\omega)$ (V²/Hz)")
ax3b.set_title("Photon Noise $N_{S_1}(\\omega)$")
ax3b.grid(True, alpha=0.3)

fig3.tight_layout()
fig3.savefig(results_dir / "noise_spectra_extracted.png", dpi=150)
print(f"提取的噪声谱已保存")

plt.show()
print(f"\n所有图表已保存至: {results_dir}")

In [ ]:
# ===== 安全断开设备 =====
print("安全断开设备...")

# 1. 关闭 X/Y 补偿输出
dev = devices.get("dg_comp")
if dev:
    dev.set_output(False, channel=1)
    dev.set_output(False, channel=2)
    print("  X/Y 补偿: 输出 OFF")
    dev.set_sync_state(False, channel=1)
    dev.set_sync_state(False, channel=2)
    dev.set_burst_state(False, channel=1)
    dev.set_burst_state(False, channel=2)

# 2. 恢复温度开关
dev = devices.get("dg_temp")
if dev:
    dev.set_output(True, channel=2)
    print("  温度开关: ON (确保温控恢复)")

# 3. 关闭 Pump 调制
dev = devices.get("dg_mod")
if dev:
    dev.set_output(False, channel=1)
    dev.set_output(False, channel=2)
    print("  Pump 调制: 输出 OFF")

# 4. 关闭 GS200 主磁场
dev = devices.get("gs200")
if dev:
    dev.set_output(False)
    print("  主磁场 GS200: 输出 OFF")

# 5. 关闭光功率
dev = devices.get("dg_laser")
if dev:
    dev.set_output(False, channel=1)
    dev.set_output(False, channel=2)
    print("  光功率: 输出 OFF")

# 6. 关闭 HF2（解调器 disable）
dev = devices.get("hf2")
if dev:
    demod.set_demod_enable(dev, 0, False)
    print("  HF2: 解调器 0 DISABLE")

# 7. 断开所有连接
for name, dev in devices.items():
    try:
        if hasattr(dev, "disconnect"):
            dev.disconnect()
            print(f"  {name}: 已断开")
    except Exception as e:
        print(f"  {name} 断开失败: {e}")

print("\n✅ 所有设备已安全断开")